# pygeoglim — Global Geology Data Fetcher

Demonstrates how to fetch raw GLiM (lithology) and GLHYMPS (hydrogeology) polygons for any watershed on Earth, and compute CAMELS-style attribute summaries.

```bash
pip install pygeoglim geopandas matplotlib
```

In [ ]:
from shapely.geometry import box
from pygeoglim import (
    fetch_glim, fetch_glhymps,
    glim_attributes, glhymps_attributes,
    decode_glim_lithology,
)

## 1. Define a watershed

Pass any `shapely` geometry in WGS-84 (lon/lat). `box(minx, miny, maxx, maxy)` is the quickest way to create a bounding-box polygon.

In [ ]:
# Rhine River headwaters — Switzerland / Germany
rhine = box(6.0, 46.5, 8.5, 48.5)

# Amazon headwaters — Peru
amazon = box(-77.0, -12.0, -73.0, -8.0)

## 2. Fetch raw lithology polygons (GLiM)

In [ ]:
# region='global' → downloads from the global sharded tiles
# region='auto'   → auto-detect (CONUS tile if centroid in CONUS, else global)
glim_gdf = fetch_glim(rhine, region="global")
print(glim_gdf.shape)
print(glim_gdf.columns.tolist())
glim_gdf.head()

In [ ]:
# Human-readable lithology names
glim_gdf["lith_name"] = glim_gdf["xx"].map(decode_glim_lithology)
glim_gdf[["xx", "lith_name", "geometry"]].head(10)

## 3. Fetch raw hydrogeology polygons (GLHYMPS)

In [ ]:
glhymps_gdf = fetch_glhymps(rhine, region="global")
print(glhymps_gdf.shape)
print(glhymps_gdf.columns.tolist())
glhymps_gdf.head()

## 4. CAMELS-style attribute summaries

In [ ]:
glim_attrs = glim_attributes(glim_gdf)
glim_attrs

In [ ]:
glhymps_attrs = glhymps_attributes(glhymps_gdf)
glhymps_attrs

## 5. Visualise lithology map

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(9, 7))

# Colour each lithology class distinctly
lith_classes = glim_gdf["xx"].unique()
cmap = plt.cm.get_cmap("tab20", len(lith_classes))
colour_map = {cls: cmap(i) for i, cls in enumerate(lith_classes)}

for cls, grp in glim_gdf.groupby("xx"):
    grp.plot(ax=ax, color=colour_map[cls], edgecolor="none", alpha=0.8)

# Watershed outline
import geopandas as gpd
gpd.GeoSeries([rhine]).plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.5)

# Legend
patches = [
    mpatches.Patch(color=colour_map[c], label=f"{c} — {decode_glim_lithology(c)}")
    for c in lith_classes
]
ax.legend(handles=patches, fontsize=7, loc="lower left", framealpha=0.9)
ax.set_title("GLiM lithology — Rhine headwaters", fontsize=13)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## 6. Visualise permeability

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

glhymps_gdf.plot(
    column="logK_Ice_x",
    ax=ax,
    cmap="RdYlBu",
    legend=True,
    legend_kwds={"label": "log₁₀ permeability (m²)", "shrink": 0.6},
    edgecolor="none",
    alpha=0.9,
)
gpd.GeoSeries([rhine]).plot(ax=ax, facecolor="none", edgecolor="black", linewidth=1.5)
ax.set_title("GLHYMPS permeability (log₁₀ m²) — Rhine headwaters", fontsize=13)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## 7. Multi-basin batch

Run for several watersheds and collect results into a single attribute table.

In [ ]:
import pandas as pd

basins = {
    "Rhine (Germany)":   box(6.0,  46.5, 8.5,  48.5),
    "Amazon headwaters": box(-77.0, -12.0, -73.0, -8.0),
    "Ganges (India)":    box(77.0,  25.0, 83.0,  29.0),
    "Mississippi (CONUS)": box(-93.0, 36.0, -88.0, 41.0),
}

rows = []
for name, geom in basins.items():
    glim_g   = fetch_glim(geom, region="auto")
    glhymps_g = fetch_glhymps(geom, region="auto")
    row = {"basin": name}
    row.update(glim_attributes(glim_g))
    row.update(glhymps_attributes(glhymps_g))
    rows.append(row)
    print(f"  ✓ {name}")

df = pd.DataFrame(rows).set_index("basin")
df